# TCN subsample run — evaluation notebook

Evaluate the prior: `bash run_tcn_baseline_160_original_instancenorm_subsample.sh --prebuilt-mmap-dir ./subseqs_original_mmap --batch-size 16`

1. Load **history.json** and plot train/val loss and val F1.
2. Load checkpoint and **validation data** (prebuilt mmap, decimate 10).
3. Visualize validation samples: **overlay GT disruption** and **predicted disruption** time steps on the signal.

**GT disruption** = first time step where the label is 1. In the pipeline this is **t_disruption − 300 ms** (Twarn), i.e. the *warning* time, not the actual disruption time. The target is built so label=1 from that point onward.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Paths — set to your run. On server use e.g.:
#   Temporary/dpark1/scratch/soen_fusion_zero/checkpoints_tcn_ddp_original/L4_H80_20260311_095201
CHECKPOINT_DIR = Path("checkpoints_tcn_ddp_original/L4_H80_20260311_095201")
PREBUILT_MMAP_DIR = "./subseqs_original_mmap"
DECIMATE_FACTOR = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load and plot training history

In [ ]:
history_path = CHECKPOINT_DIR / "history.json"
if not history_path.exists():
    raise FileNotFoundError(f"Not found: {history_path}")

with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(epochs, history["train_loss"], label="Train")
axes[0, 0].plot(epochs, history["val_loss"], label="Val")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, history["val_f1"], color="C2")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("F1")
axes[0, 1].set_title("Val F1")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs, history["val_acc"], label="Val acc@th")
axes[1, 0].plot(epochs, history["val_precision"], label="Precision")
axes[1, 0].plot(epochs, history["val_recall"], label="Recall")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Metric")
axes[1, 0].set_title("Val accuracy / P / R")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs, history["val_threshold"], color="C4")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Threshold")
axes[1, 1].set_title("Val best threshold")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("TCN subsample run — training history", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Load model and validation dataset

In [ ]:
# Import model builder and dataset (same as train_tcn_ddp_original)
from disruptcnn.dataset_original import PrebuiltOriginalSubseqDataset
from train_tcn_ddp_original import build_model

ckpt_path = CHECKPOINT_DIR / "best.pt"
if not ckpt_path.exists():
    ckpt_path = CHECKPOINT_DIR / "last.pt"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

args = state["args"]
nrecept = state["nrecept"]

model, _, _ = build_model(
    args["input_channels"], 1, args["levels"], args["nhid"],
    args["kernel_size"], args["dilation_base"], args["dropout"],
    nrecept_target=args["nrecept_target"],
    use_instance_norm=args.get("use_instance_norm", False),
    use_prenorm=args.get("use_prenorm", False),
)
model.load_state_dict(state["state_dict"])
model = model.to(DEVICE)
model.eval()

ds = PrebuiltOriginalSubseqDataset(PREBUILT_MMAP_DIR, decimate_factor=DECIMATE_FACTOR)
val_inds = ds.get_split_indices("val")
if len(val_inds) == 0:
    val_inds = ds.get_split_indices("test")

print(f"Checkpoint: {ckpt_path}")
print(f"nrecept: {nrecept}, val samples: {len(val_inds)}")

## 3. Run inference and get GT / predicted disruption timesteps

In [ ]:
def first_disrupt_step(target: np.ndarray) -> int:
    """First time step index where target >= 0.5. Return -1 if no disruption."""
    pos = np.where(np.asarray(target).ravel() >= 0.5)[0]
    return int(pos[0]) if len(pos) > 0 else -1


def predict_disrupt_step(model, x: torch.Tensor, nrecept: int, device: torch.device, thresh: float = 0.5) -> int:
    """
    x: (1, C, T). Returns first *sequence* time step where pred >= thresh.
    Model output is valid from index nrecept-1; pred index 0 -> sequence step nrecept-1.
    """
    with torch.no_grad():
        out = model(x)
    # out (1, T) from TCN; valid predictions from index nrecept-1
    pred = out[0, nrecept - 1:].cpu().numpy()
    pos = np.where(pred >= thresh)[0]
    if len(pos) == 0:
        return -1
    return int(nrecept - 1 + pos[0])


results = []
n_vis = min(12, len(val_inds))
indices_to_plot = np.linspace(0, len(val_inds) - 1, n_vis, dtype=int)

for idx in indices_to_plot:
    i = val_inds[idx]
    x, target, weight = ds[i]
    x = x.unsqueeze(0).to(DEVICE)
    # Flatten channel dims: (1, C, T) or (1, 20, 8, T) -> (1, 160, T); TCN expects (B, C, T)
    x = x.view(1, -1, x.shape[-1])
    gt_step = first_disrupt_step(target.numpy())
    pred_step = predict_disrupt_step(model, x, nrecept, DEVICE)
    with torch.no_grad():
        out = model(x)
    pred_dense = out[0, nrecept - 1:].cpu().numpy()  # (T_valid,) probability per step
    results.append({
        "val_idx": idx,
        "seq_idx": i,
        "x": x.cpu(),
        "target": target,
        "gt_step": gt_step,
        "pred_step": pred_step,
        "pred_dense": pred_dense,
    })

print(f"Computed GT and predicted disruption steps for {len(results)} validation samples.")

## 4. Visualize: signal with overlaid GT and predicted disruption time steps

In [ ]:
def plot_signal_gt_pred(r, ax, signal="mean"):
    """
    Plot one validation sample: signal vs time, vertical lines at GT and predicted disruption.
    r: dict with x (1,C,T), target, gt_step, pred_step.
    signal: 'mean' (mean over channels) or 'first' (first channel).
    """
    x = r["x"][0].numpy()  # (C, T)
    if signal == "mean":
        sig = x.mean(axis=0)
    else:
        sig = x[0]
    T = sig.shape[0]
    ax.plot(np.arange(T), sig, color="#333", linewidth=0.6, alpha=0.9, label="Signal (mean ch)")
    gt = r["gt_step"]
    pred = r["pred_step"]
    if gt >= 0:
        ax.axvline(gt, color="green", linestyle="-", linewidth=1.5, label="GT disruption")
    if pred >= 0:
        ax.axvline(pred, color="red", linestyle="--", linewidth=1.5, label="Pred disruption")
    ax.set_xlabel("Time (decimated step)")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"Val idx {r['val_idx']} (seq {r['seq_idx']})")


n_cols = 3
n_rows = (n_vis + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
for k, r in enumerate(results):
    i, j = k // n_cols, k % n_cols
    plot_signal_gt_pred(r, axes[i, j])
for k in range(len(results), n_rows * n_cols):
    i, j = k // n_cols, k % n_cols
    axes[i, j].set_visible(False)

plt.suptitle("Validation samples: signal with GT (green) and predicted (red) disruption time step", fontsize=11)
plt.tight_layout()
plt.savefig("evaluation_subsample_gt_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Dense labeling: GT and predicted probability over time (where mistakes lie)

Plot the **full per-timestep label** (GT) and **model probability** (pred_dense), not just the first-hit bar. This shows where the model wrongly fires on **clear** subsequences (false positives) or misses on disruptive ones (false negatives).

In [ ]:
THRESH = 0.5

def plot_dense_labels(r, ax, nrecept, thresh=0.5):
    """
    Plot dense GT label (0/1) and dense predicted probability. Shade false positives (GT=0, pred>=thresh)
    and false negatives (GT=1, pred<thresh).
    """
    target = np.asarray(r["target"]).ravel()
    pred_dense = r["pred_dense"]
    # Predictions are valid from step nrecept-1; align target slice
    start = nrecept - 1
    target_valid = target[start:]
    T = len(pred_dense)
    if len(target_valid) != T:
        target_valid = target_valid[:T]
    steps = np.arange(T) + start

    ax.fill_between(steps, 0, target_valid, color="green", alpha=0.25, label="GT label (1=disrupt)")
    ax.plot(steps, pred_dense, color="blue", linewidth=0.8, label="Pred prob")
    ax.axhline(thresh, color="gray", linestyle="--", linewidth=0.8)

    # False positives: GT=0 but pred >= thresh (model fires on clear)
    fp = (target_valid < 0.5) & (pred_dense >= thresh)
    if np.any(fp):
        ax.fill_between(steps, 0, 1, where=fp, color="red", alpha=0.35, label="FP (mistake on clear)")
    # False negatives: GT=1 but pred < thresh (model misses disruption)
    fn = (target_valid >= 0.5) & (pred_dense < thresh)
    if np.any(fn):
        ax.fill_between(steps, 0, 1, where=fn, color="orange", alpha=0.35, label="FN (missed disrupt)")

    ax.set_xlabel("Time (decimated step)")
    ax.set_ylabel("Label / Prob")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="upper right", fontsize=6)
    ax.grid(True, alpha=0.3)
    is_clear = r["gt_step"] < 0
    title = f"Val {r['val_idx']} (seq {r['seq_idx']}) — {'CLEAR' if is_clear else 'disrupt'}"
    if is_clear and r["pred_step"] >= 0:
        title += " [FP: model fired]"
    ax.set_title(title)


n_cols = 3
n_rows = (n_vis + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
for k, r in enumerate(results):
    i, j = k // n_cols, k % n_cols
    plot_dense_labels(r, axes[i, j], nrecept, THRESH)
for k in range(len(results), n_rows * n_cols):
    i, j = k // n_cols, k % n_cols
    axes[i, j].set_visible(False)

plt.suptitle("Dense labeling: GT (green fill) vs pred prob (blue). Red=FP (mistake on clear), Orange=FN (missed disrupt)", fontsize=10)
plt.tight_layout()
plt.savefig("evaluation_subsample_dense_labels.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Fine-grained evaluation: confusion matrix and grouped majority vote

**Why F1 here was below training's val_f1?** (1) Training reports F1 at the **best** threshold (sweep 0.05–0.95). (2) **best.pt** is saved at the **best F1 epoch**, so we must use that epoch's threshold (e.g. 0.6), not the *last* epoch's (e.g. 0.35). The notebook now uses the threshold from the best F1 epoch in history.

1. **Threshold sweep**: Same as training — sweep 19 thresholds, set THRESH = argmax F1.
2. **Confusion matrix** (per-step, M=1 at that threshold): TP, TN, FP, FN and metrics.
3. **Group size sweep**: Group steps into windows of size M, majority-vote; find **best M** by F1.
4. **pred_step with best M**: First group where majority = 1; error distribution and overlays.

In [ ]:
# Run inference on full validation set for fine-grained eval (optional limit to avoid long runtime)
MAX_VAL_SAMPLES = min(500, len(val_inds))  # set to len(val_inds) for full set
val_indices_eval = val_inds[:MAX_VAL_SAMPLES]
full_val_data = []
start = nrecept - 1

for idx, i in enumerate(val_indices_eval):
    x, target, weight = ds[i]
    x = x.unsqueeze(0).to(DEVICE).view(1, -1, x.shape[-1])
    with torch.no_grad():
        out = model(x)
    pred_dense = out[0, start:].cpu().numpy()
    target_flat = np.asarray(target).ravel()
    target_valid = target_flat[start : start + len(pred_dense)]
    if len(target_valid) < len(pred_dense):
        pred_dense = pred_dense[: len(target_valid)]
    gt_step = first_disrupt_step(target_flat)
    full_val_data.append({
        "pred_dense": pred_dense,
        "target_valid": target_valid,
        "gt_step": gt_step,
        "seq_idx": i,
        "val_idx": idx,
    })


In [ ]:
# Match training's F1: train_tcn_ddp_original.py sweeps thresholds np.linspace(0.05, 0.95, 19) and reports F1 at the best.
# The 0.8750 performance was at threshold 0.6 (one of the 19). Prefer history's threshold when available.
y_true_all = np.concatenate([d["target_valid"] for d in full_val_data])
y_pred_raw = np.concatenate([d["pred_dense"] for d in full_val_data])
y_true_bin = (y_true_all >= 0.5).astype(np.int32)

thresholds = np.linspace(0.05, 0.95, 19)  # same as train_tcn_ddp_original.py evaluate()
best_f1, best_th = 0.0, 0.5
for th in thresholds:
    yp = (y_pred_raw >= th).astype(np.int32)
    tp = ((yp == 1) & (y_true_bin == 1)).sum()
    fp = ((yp == 1) & (y_true_bin == 0)).sum()
    fn = ((yp == 0) & (y_true_bin == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    if f1 > best_f1:
        best_f1, best_th = f1, th

# Use threshold from the *best* F1 epoch (best.pt is saved at that epoch), not the last epoch.
if history and "val_threshold" in history and "val_f1" in history:
    best_epoch_idx = int(np.argmax(history["val_f1"]))
    train_th = history["val_threshold"][best_epoch_idx]
    train_f1 = history["val_f1"][best_epoch_idx]
    THRESH = float(train_th)
    print(f"Using threshold from best F1 epoch (epoch {best_epoch_idx + 1}): {THRESH:.2f}  (val_f1={train_f1:.4f})")
else:
    THRESH = float(best_th)
    print(f"No history; using sweep best: THRESH={THRESH:.2f}, F1={best_f1:.4f}")
print(f"Sweep (this val set):   best F1={best_f1:.4f} at threshold={best_th:.2f}")
print(f"Collected dense pred/target for {len(full_val_data)} validation samples.")

In [ ]:
# Confusion matrix (per-step, M=1): all valid timesteps across validation samples
y_true = np.concatenate([d["target_valid"] for d in full_val_data])
y_pred = (np.concatenate([d["pred_dense"] for d in full_val_data]) >= THRESH).astype(np.int32)
# Binarize GT if needed (target may be 0/1 or float)
y_true = (y_true >= 0.5).astype(np.int32)

tp = int(((y_pred == 1) & (y_true == 1)).sum())
tn = int(((y_pred == 0) & (y_true == 0)).sum())
fp = int(((y_pred == 1) & (y_true == 0)).sum())
fn = int(((y_pred == 0) & (y_true == 1)).sum())
n = len(y_true)
acc = (tp + tn) / n if n else 0
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print(f"Per-step (M=1) — Confusion matrix (threshold = {THRESH:.2f} from sweep)")
print(f"                Pred=0   Pred=1")
print(f"  True=0 (clear)  {tn:7d}   {fp:7d}")
print(f"  True=1 (disrupt) {fn:6d}   {tp:7d}")
print(f"\n  Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")

# Heatmap
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
cm = np.array([[tn, fp], [fn, tp]])
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred=0", "Pred=1"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True=0 (clear)", "True=1 (disrupt)"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
ax.set_title("Confusion matrix (per-step, M=1)")
plt.tight_layout()
plt.show()

In [ ]:
def majority_vote_group(labels: np.ndarray, M: int) -> tuple:
    """Group consecutive steps into windows of M; majority vote per group. Returns (group_pred, group_target) arrays."""
    n = len(labels)
    n_groups = (n + M - 1) // M
    pred_bin = (labels["pred"] >= THRESH).astype(np.int32)
    targ_bin = (labels["target"] >= 0.5).astype(np.int32)
    p_g, t_g = [], []
    for g in range(n_groups):
        lo, hi = g * M, min((g + 1) * M, n)
        if lo >= hi:
            continue
        p_g.append(1 if np.sum(pred_bin[lo:hi]) > (hi - lo) / 2 else 0)
        t_g.append(1 if np.sum(targ_bin[lo:hi]) > (hi - lo) / 2 else 0)
    return np.array(p_g), np.array(t_g)

def metrics_from_binary(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    n = len(y_true)
    acc = (tp + tn) / n if n else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "tp": tp, "tn": tn, "fp": fp, "fn": fn}

# Sweep group sizes M
M_list = [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 50]
sweep_results = []
for M in M_list:
    all_p, all_t = [], []
    for d in full_val_data:
        pred_d = d["pred_dense"]
        targ_d = d["target_valid"]
        L = min(len(pred_d), len(targ_d))
        struct = np.zeros(L, dtype=[("pred", np.float32), ("target", np.float32)])
        struct["pred"] = pred_d[:L]
        struct["target"] = targ_d[:L]
        p_g, t_g = majority_vote_group(struct, M)
        all_p.append(p_g)
        all_t.append(t_g)
    y_pred_m = np.concatenate(all_p)
    y_true_m = np.concatenate(all_t)
    m = metrics_from_binary(y_true_m, y_pred_m)
    m["M"] = M
    sweep_results.append(m)

# Best M by F1
best = max(sweep_results, key=lambda x: x["f1"])
best_M = best["M"]
print(f"Best group size M = {best_M} (F1 = {best['f1']:.4f})")
print("\nMetrics by M:")
print(f"{'M':>4}  {'Accuracy':>8}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
for r in sweep_results:
    mark = " *" if r["M"] == best_M else ""
    print(f"{r['M']:>4}  {r['accuracy']:>8.4f}  {r['precision']:>10.4f}  {r['recall']:>8.4f}  {r['f1']:>8.4f}{mark}")

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot([r["M"] for r in sweep_results], [r["f1"] for r in sweep_results], "o-", label="F1")
ax.axvline(best_M, color="gray", linestyle="--", label=f"best M={best_M}")
ax.set_xlabel("Group size M")
ax.set_ylabel("F1 score")
ax.set_title("F1 vs group size (majority vote per group)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# pred_step with best M: first *group* where majority vote is 1 -> step = start of that group
def first_disrupt_step_grouped(pred_dense: np.ndarray, M: int, start: int, thresh: float = 0.5) -> int:
    """First sequence step of the first group (of size M) where majority of pred >= thresh. -1 if none."""
    pred_bin = (pred_dense >= thresh).astype(np.int32)
    n = len(pred_bin)
    for g in range((n + M - 1) // M):
        lo, hi = g * M, min((g + 1) * M, n)
        if lo >= hi:
            continue
        if np.sum(pred_bin[lo:hi]) > (hi - lo) / 2:
            return start + g * M
    return -1

results_grouped = []
for d in full_val_data:
    pred_step_g = first_disrupt_step_grouped(d["pred_dense"], best_M, start, THRESH)
    results_grouped.append({
        **d,
        "pred_step_grouped": pred_step_g,
    })

# Error distribution (grouped): pred_step_grouped - gt_step (only when both disruptive)
errors_grouped = []
for r in results_grouped:
    if r["gt_step"] >= 0 and r["pred_step_grouped"] >= 0:
        errors_grouped.append(r["pred_step_grouped"] - r["gt_step"])

print(f"Using best M = {best_M}: {len(errors_grouped)} disruptive samples with both GT and predicted disruption.")

if errors_grouped:
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.hist(errors_grouped, bins=min(30, len(set(errors_grouped)) + 1), edgecolor="black", alpha=0.7)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_xlabel("Predicted disruption step − GT step (grouped, M=%d)" % best_M)
    ax.set_ylabel("Count")
    ax.set_title("Disruption timing error (validation, disruptive only, majority-vote group size M=%d)" % best_M)
    plt.tight_layout()
    plt.show()
else:
    print("No disruptive samples with predicted disruption in this subset.")

In [ ]:
# Overlay pred_step (grouped, best M) on the same signal subset as in section 4
lookup = {r["seq_idx"]: r["pred_step_grouped"] for r in results_grouped}
results_with_grouped = []
for r in results:
    pred_g = lookup.get(r["seq_idx"], -1)
    results_with_grouped.append({**r, "pred_step_grouped": pred_g})

n_cols = 3
n_rows = (n_vis + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
for k, r in enumerate(results_with_grouped):
    i, j = k // n_cols, k % n_cols
    ax = axes[i, j]
    x = r["x"][0].numpy()
    sig = x.mean(axis=0)
    T = sig.shape[0]
    ax.plot(np.arange(T), sig, color="#333", linewidth=0.6, alpha=0.9, label="Signal (mean ch)")
    gt = r["gt_step"]
    if gt >= 0:
        ax.axvline(gt, color="green", linestyle="-", linewidth=1.5, label="GT disruption")
    pred_g = r["pred_step_grouped"]
    if pred_g >= 0:
        ax.axvline(pred_g, color="red", linestyle="--", linewidth=1.5, label="Pred (grouped M=%d)" % best_M)
    ax.set_xlabel("Time (decimated step)")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right", fontsize=6)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"Val idx {r['val_idx']} (seq {r['seq_idx']})")
for k in range(len(results_with_grouped), n_rows * n_cols):
    i, j = k // n_cols, k % n_cols
    axes[i, j].set_visible(False)
plt.suptitle("Validation samples: signal with GT (green) and predicted disruption step (grouped, M=%d)" % best_M, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Clear subsequences only: where do mistakes (false positives) lie?
clear_results = [r for r in results if r["gt_step"] < 0]
print(f"Among the {len(results)} plotted validation samples, {len(clear_results)} are CLEAR (no disruption).")
if not clear_results:
    print("No clear samples in this subset. Run with more val indices or check split.")
else:
    for r in clear_results:
        pred_dense = r["pred_dense"]
        target = np.asarray(r["target"]).ravel()
        start = nrecept - 1
        target_valid = target[start:start + len(pred_dense)]
        fp_mask = (target_valid < 0.5) & (pred_dense >= THRESH)
        n_fp = np.sum(fp_mask)
        if n_fp > 0:
            fp_steps = np.where(fp_mask)[0] + start
            print(f"  Val {r['val_idx']} (seq {r['seq_idx']}): CLEAR but model fired at {n_fp} steps — first FP at step {fp_steps[0]}, last at {fp_steps[-1]}")
        else:
            print(f"  Val {r['val_idx']} (seq {r['seq_idx']}): CLEAR, no false positives ✓")

    # Plot dense labels for clear-only samples (mistakes highlighted)
    n_clear = len(clear_results)
    n_rows_c = max(1, (n_clear + 1) // 2)
    n_cols_c = min(2, n_clear)
    fig2, axes2 = plt.subplots(n_rows_c, n_cols_c, figsize=(5 * n_cols_c, 3 * n_rows_c))
    if n_clear == 1:
        axes2 = np.array([axes2])
    else:
        axes2 = axes2.ravel()
    for k, r in enumerate(clear_results):
        plot_dense_labels(r, axes2[k], nrecept, THRESH)
    for k in range(len(clear_results), len(axes2)):
        axes2[k].set_visible(False)
    plt.suptitle("Clear subsequences: dense GT (green) vs pred (blue). Red = false positive (model wrongly fired)", fontsize=10)
    plt.tight_layout()
    plt.savefig("evaluation_subsample_clear_mistakes.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Optional: error distribution (raw pred_step, M=1) for comparison

**pred_step** = first sequence time step where the model output ≥ 0.5 (from `predict_disrupt_step`). Errors are pred_step − gt_step (early → negative, late → positive).

In [ ]:
errors = []
for r in results:
    if r["gt_step"] >= 0 and r["pred_step"] >= 0:
        errors.append(r["pred_step"] - r["gt_step"])

if errors:
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.hist(errors, bins=min(30, len(set(errors)) + 1), edgecolor="black", alpha=0.7)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_xlabel("Predicted disruption step − GT step")
    ax.set_ylabel("Count")
    ax.set_title("Disruption timing error (validation, disruptive only)")
    plt.tight_layout()
    plt.show()
else:
    print("No disruptive samples in the plotted subset to compute errors.")